# The Robert and Rosenbaum Uncertainty Zones model

## Implementation by 
## Marcos Costa Santos Carreira
## École Polytechnique - CMAP
## Dec-2019

## Import packages

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from functools import partial
import scipy.stats as st
import seaborn as sns
from scipy import optimize

In [2]:
# import ruptures as rpt

In [3]:
# import datetime as dtt
import timeit

In [4]:
import uz as uz

In [5]:
#import cProfile
# useful to check bottlenecks
# But the biggest improvement is to parallelize \
#     the Monte Carlo simulations

In [6]:
pd.set_option('display.max_columns', 80)

## File paths

In [7]:
# pathdata='/Users/marcoscscarreira/Documents/X/CME project/data/'
# pathdfs='/Users/marcoscscarreira/Documents/X/CME project/dfs/'
pathdfs='/Users/marcoscscarreira/My Papers/UZModelUncertainty/dfs/'
# useful to store the raw paths, the processed paths and the statistics

## Simulations

### Parameters

In [8]:
# npaths = 50 # should be enough?
# npaths = 2
# Spot = 100.
# tm = 1. # one trading day
# dt = 0.005 # step 0.005s=5ms
# hours = 8 # a trading day for stocks
# dtosec = np.sqrt(hours*3600)
# nrsteps = int(np.round((dtosec**2)/dt))
# tmrst = uz.atrange(nrsteps,tm)

In [9]:
npaths = 1
s = 100.
tm = 1. # one trading day
dt = 0.005 # step 0.005s=5ms
nrsteps = 2**23 # 2**23
hours = dt*(nrsteps)/3600 # a trading day for stocks
dtosec = np.sqrt(hours*3600)
tmrst = uz.atrange(nrsteps, tm)
hours = 9
tmrstsec = tmrst * hours * 3600

In [10]:
# vollist   = [0.00125, 0.0025, 0.005, 0.01, 0.02] # For the trading hours
# alphalist = [0.005, 0.01, 0.02, 0.04]
# etalist   = [0.0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
# mulist = [-0.03, -0.02, -0.01, -0.005, 0.0, +0.005, +0.01, +0.02, +0.03]
# spotlist = [97., 98., 99., 99.5, 99.75, 100., 100.25, 100.5, 101., 102., 103.]

The durations can be estimated by the formula:

$2\cdot\eta\cdot\left(\frac{\alpha}{S}\cdot\frac{1}{\sigma}\right)^{2}$

In [11]:
# durlist = np.array([[v, a, e, uz.dur(Spot, a, e, v)] \
#                        for e in etalist \
#                        for a in alphalist \
#                        for v in vollist])

In [12]:
# durdf=pd.DataFrame(durlist,columns=['vol','alpha','eta','dur'])
# durdf['dursec']=durdf['dur']*dtosec**2

We want to make sure our simulation has steps small enough to capture
the durations for $\eta$=0.1

In [13]:
# durdf[durdf['eta']==0.1].set_index(['vol','alpha','eta']).sort_index()

In [14]:
# durdf.set_index(['vol','alpha','eta']).sort_index()

### Raw paths - Generation

In [15]:
seed = 42

In [16]:
rndns = uz.frndn(nrsteps, seed)

In [17]:

σ = 0.01
μ = 0.
σ2 = 0.02
μ2 = 0.05
σ3 = 0.05
μ3 = 0.0004*(9*60)
v = np.full(nrsteps, σ)
mu = np.full(nrsteps, μ)
v1 = np.concatenate((np.full(nrsteps//2, σ), np.full(nrsteps//2, σ2)))
mu1 = np.concatenate((np.full(nrsteps//2, μ), np.full(nrsteps//2, μ2)))
v2 = np.concatenate((np.full(nrsteps//2-500000, σ),
                     np.full(1000000, σ3),
                     np.full(nrsteps//2-500000, σ)))
mu2 = np.concatenate((np.full(nrsteps//2-500000, μ),
                      np.full(1000000, μ3),
                      np.full(nrsteps//2-500000, μ)))

In [18]:
# pd.Series(v, index=tmrst[:-1]).plot(figsize=(9, 6), color='k');
# pd.Series(v1, index=tmrst[:-1]).plot(figsize=(9, 6), color='g');
# pd.Series(v2, index=tmrst[:-1]).plot(figsize=(9, 6), color='b');

In [19]:
# pd.Series(mu, index=tmrst[:-1]).plot(figsize=(9, 6), color='k');
# pd.Series(mu1, index=tmrst[:-1]).plot(figsize=(9, 6), color='g');
# pd.Series(mu2, index=tmrst[:-1]).plot(figsize=(9, 6), color='b');

In [20]:
# tmrst = uz.atrange(nrsteps, tm)
# tmrst = uz.atrange(nrsteps, tm)

In [21]:
path0 = uz.MCPath(rndns, v, s, tm, mu)  # Black, trend 0 vol 0.01
path1 = uz.MCPath(rndns, v1, s, tm, mu)  # Green, vol from 0.01 to 0.02
path2 = uz.MCPath(rndns, v, s, tm, mu1)  # Blue, trend from 0 to 0.05
path3 = uz.MCPath(rndns, v1, s, tm, mu1)  # Magenta, both changes
path4 = uz.MCPath(rndns, v, s, tm, mu2)  # Red, burst
path5 = uz.MCPath(rndns, v2, s, tm, mu2)  # Yellow, burst with vol

In [22]:
# pd.Series(path0, index=tmrst).plot(figsize=(9, 6), color='k');
# pd.Series(path1, index=tmrst).plot(figsize=(9, 6), color='g');
# pd.Series(path2, index=tmrst).plot(figsize=(9, 6), color='b');
# pd.Series(path3, index=tmrst).plot(figsize=(9, 6), color='m');
# pd.Series(path4, index=tmrst).plot(figsize=(9, 6), color='r');
# pd.Series(path5, index=tmrst).plot(figsize=(9, 6), color='y');

In [23]:

# signal = np.transpose(np.array([path0, path1, path2, path3]))

In [24]:

# algo = rpt.Pelt(model='l2', min_size=100).fit(signal)
# my_bkps = algo.predict(pen=10)

In [25]:

# rpt.display(signal, my_bkps)

In [26]:

α = 0.01
η = 0.3
trpath0 = uz.trprpath(α, η, path0)
trpath1 = uz.trprpath(α, η, path1)
trpath2 = uz.trprpath(α, η, path2)
trpath3 = uz.trprpath(α, η, path3)
trpath4 = uz.trprpath(α, η, path4)
trpath5 = uz.trprpath(α, η, path5)

In [27]:
# pd.Series(trpath0, index=tmrst).plot(figsize=(9, 6), color='k');
# pd.Series(trpath1, index=tmrst).plot(figsize=(9, 6), color='g');
# pd.Series(trpath2, index=tmrst).plot(figsize=(9, 6), color='b');
# pd.Series(trpath3, index=tmrst).plot(figsize=(9, 6), color='m');
# pd.Series(trpath4, index=tmrst).plot(figsize=(9, 6), color='r');
# pd.Series(trpath5, index=tmrst).plot(figsize=(9, 6), color='y');

In [28]:

# signal_tr = np.transpose(np.array([trpath0, trpath1, trpath2, trpath3]))

In [29]:

# algo_tr = rpt.Pelt(model='l2', min_size=100).fit(signal_tr)
# my_bkps_tr = algo_tr.predict(pen=3)

In [30]:

# rpt.display(signal_tr, my_bkps_tr)

In [31]:

pxchg0 = uz.diff_prices_df_nogroup(pd.Series(trpath0, index=tmrstsec), α)
pxchg1 = uz.diff_prices_df_nogroup(pd.Series(trpath1, index=tmrstsec), α)
pxchg2 = uz.diff_prices_df_nogroup(pd.Series(trpath2, index=tmrstsec), α)
pxchg3 = uz.diff_prices_df_nogroup(pd.Series(trpath3, index=tmrstsec), α)
pxchg4 = uz.diff_prices_df_nogroup(pd.Series(trpath4, index=tmrstsec), α)
pxchg5 = uz.diff_prices_df_nogroup(pd.Series(trpath5, index=tmrstsec), α)

In [32]:
uz.distance_dur(pxchg0, s, α, η, σ, μ, hours)

0.4552553260013474

In [33]:
hour_range = range(1, hours + 1)
min_range = range(10, hours*60 + 10, 10)

In [34]:
# start = timeit.default_timer()
# res0de = uz.minde_dist(pxchg0, s, α, hours)
# stop = timeit.default_timer()
# print('Time Spent: ', round(stop - start), ' seconds')
# df0_res = pd.DataFrame([res0de], index=[0],
#                        columns=['H', 'η', 'σ', 'σXe', 'σP', 'μ', 'μmax', 'distance'])
# print(df0_res)

In [35]:
# start = timeit.default_timer()
# res0de = uz.minde_dist(pxchg0, s, α, hours, 0.01)
# stop = timeit.default_timer()
# print('Time Spent: ', round(stop - start), ' seconds')
# df0_res = pd.DataFrame([res0de], index=[0],
#                        columns=['H', 'η', 'σ', 'σXe', 'σP', 'μ', 'μmax', 'distance'])
# print(df0_res)

In [42]:
start = timeit.default_timer()
res0dmf = uz.multfit_dist(pxchg0, s, α, hours, 'All', 0)
stop = timeit.default_timer()
print('Time Spent: ', round(stop - start), ' seconds')

Time Spent:  87  seconds


In [43]:
res0dmf.T

,All
Al_Up,0.304947
Co_Up,0.194110
Al_Do,0.305012
Co_Do,0.195931
Al,0.609959
Co,0.390041
Al_Co_dist,0.114590
H,0.319727
Al_Up_η,0.317355
Co_Up_η,0.301549


In [44]:
start = timeit.default_timer()
res0_tbl = [uz.multfit_dist(pxchg0.loc[:h*3600], s, α, hours, h)
             for h in hour_range]
stop = timeit.default_timer()
print('Time Spent: ', round(stop - start), ' seconds')

In [45]:
df0_rest = pd.concat(res0_tbl, axis=0)

In [46]:
df0_rest.T

,1,2,3,4,5,6,7,8,9
Al_Up,0.294560,0.300530,0.304450,0.303680,0.304215,0.305063,0.304140,0.305171,0.304947
Co_Up,0.196759,0.188863,0.188720,0.191961,0.193784,0.193125,0.194184,0.193839,0.194110
Al_Do,0.294560,0.300825,0.304645,0.303827,0.304332,0.305063,0.304140,0.305244,0.305012
Co_Do,0.214120,0.209782,0.202186,0.200532,0.197669,0.196749,0.197536,0.195746,0.195931
Al,0.589120,0.601355,0.609094,0.607507,0.608547,0.610126,0.608280,0.610414,0.609959
Co,0.410880,0.398645,0.390906,0.392493,0.391453,0.389874,0.391720,0.389586,0.390041
Al_Co_dist,0.318014,0.261947,0.206255,0.239160,0.222247,0.166668,0.147321,0.117999,0.114589
H,0.348723,0.331455,0.320891,0.323036,0.321629,0.319502,0.321990,0.319116,0.319727
Al_Up_η,0.317661,0.308099,0.304759,0.313593,0.317305,0.317112,0.311655,0.311855,0.317232
Co_Up_η,0.242683,0.322270,0.289011,0.274193,0.264850,0.282048,0.296442,0.297178,0.301652


In [48]:
start = timeit.default_timer()
res0_rtbl = [uz.multfit_dist(pxchg0.loc[(h-1)*3600:h*3600], s, α, hours, h)
             for h in hour_range]
stop = timeit.default_timer()
print('Time Spent: ', round(stop - start), ' seconds')
df0_rrest = pd.concat(res0_rtbl, axis=0)

Time Spent:  763  seconds


In [49]:
df0_rrest.T

,1,2,3,4,5,6,7,8,9
Al_Up,0.294560,0.306723,0.312139,0.301278,0.306312,0.309260,0.298664,0.312390,0.303204
Co_Up,0.196759,0.180672,0.188439,0.202069,0.200926,0.189866,0.200465,0.191427,0.196224
Al_Do,0.294560,0.307323,0.312139,0.301278,0.306312,0.308678,0.298664,0.312977,0.303204
Co_Do,0.214120,0.205282,0.187283,0.195374,0.186450,0.192196,0.202208,0.183206,0.197368
Al,0.589120,0.614046,0.624277,0.602556,0.612623,0.617938,0.597327,0.625367,0.606407
Co,0.410880,0.385954,0.375723,0.397444,0.387377,0.382062,0.402673,0.374633,0.393593
Al_Co_dist,0.318013,0.349974,0.361536,0.378848,0.223526,0.258765,0.264102,0.277973,0.244618
H,0.348723,0.314272,0.300926,0.329798,0.316163,0.309142,0.337062,0.299531,0.324528
Al_Up_η,0.317640,0.269717,0.287000,0.329020,0.343324,0.335028,0.293006,0.356774,0.368401
Co_Up_η,0.244754,0.330574,0.174479,0.228863,0.252492,0.361110,0.368876,0.338325,0.355127


In [50]:
start = timeit.default_timer()
res1_rtbl = [uz.multfit_dist(pxchg1.loc[(h-1)*3600:h*3600], s, α, hours, h)
             for h in hour_range]
stop = timeit.default_timer()
print('Time Spent: ', round(stop - start), ' seconds')
df1_rrest = pd.concat(res1_rtbl, axis=0)

Time Spent:  805  seconds


In [51]:
df1_rrest.T

,1,2,3,4,5,6,7,8,9
Al_Up,0.294560,0.306723,0.312139,0.301278,0.296117,0.303219,0.298760,0.298549,0.296954
Co_Up,0.196759,0.180672,0.188439,0.202069,0.208981,0.196326,0.200775,0.203556,0.202663
Al_Do,0.294560,0.307323,0.312139,0.301278,0.296117,0.303067,0.298760,0.298705,0.296954
Co_Do,0.214120,0.205282,0.187283,0.195374,0.198786,0.197388,0.201705,0.199189,0.203429
Al,0.589120,0.614046,0.624277,0.602556,0.592233,0.606286,0.597519,0.597255,0.593908
Co,0.410880,0.385954,0.375723,0.397444,0.407767,0.393714,0.402481,0.402745,0.406092
Al_Co_dist,0.318014,0.349979,0.361536,0.378847,0.346732,0.029414,0.036810,0.020753,0.027246
H,0.348723,0.314272,0.300926,0.329798,0.344262,0.324693,0.336793,0.337164,0.341881
Al_Up_η,0.317659,0.269894,0.287095,0.329029,0.188978,0.358603,0.364825,0.342533,0.332642
Co_Up_η,0.245089,0.330768,0.293669,0.229166,0.500000,0.366029,0.352007,0.348716,0.351695


In [52]:
start = timeit.default_timer()
res2_rtbl = [uz.multfit_dist(pxchg2.loc[(h-1)*3600:h*3600], s, α, hours, h)
             for h in hour_range]
stop = timeit.default_timer()
print('Time Spent: ', round(stop - start), ' seconds')
df2_rrest = pd.concat(res2_rtbl, axis=0)

Time Spent:  707  seconds


In [53]:
df2_rrest.T

,1,2,3,4,5,6,7,8,9
Al_Up,0.294560,0.306723,0.312139,0.301278,0.304142,0.304828,0.308934,0.307823,0.305223
Co_Up,0.196759,0.180672,0.188439,0.202069,0.211243,0.210588,0.206340,0.212018,0.209467
Al_Do,0.294560,0.307323,0.312139,0.301278,0.304142,0.304828,0.308357,0.308390,0.305223
Co_Do,0.214120,0.205282,0.187283,0.195374,0.180473,0.179756,0.176369,0.171769,0.180087
Al,0.589120,0.614046,0.624277,0.602556,0.608284,0.609657,0.617291,0.616213,0.610446
Co,0.410880,0.385954,0.375723,0.397444,0.391716,0.390343,0.382709,0.383787,0.389554
Al_Co_dist,0.318014,0.349976,0.361537,0.378846,0.273980,0.379201,0.230683,0.170574,0.209341
H,0.348723,0.314272,0.300926,0.329798,0.321984,0.320134,0.309991,0.311408,0.319073
Al_Up_η,0.317554,0.269801,0.286994,0.328892,0.253602,0.220320,0.323189,0.330880,0.348673
Co_Up_η,0.243773,0.331147,0.174292,0.231961,0.243767,0.281588,0.163963,0.401892,0.371966


In [54]:
start = timeit.default_timer()
res4_rtbl = [uz.multfit_dist(pxchg4.loc[(h-1)*3600:h*3600], s, α, hours, h)
             for h in hour_range]
stop = timeit.default_timer()
print('Time Spent: ', round(stop - start), ' seconds')
df4_rrest = pd.concat(res4_rtbl, axis=0)

Time Spent:  722  seconds


In [55]:
df4_rrest.T

,1,2,3,4,5,6,7,8,9
Al_Up,0.294560,0.306723,0.312139,0.301693,0.305416,0.306622,0.309642,0.311761,0.304005
Co_Up,0.196759,0.180672,0.188439,0.204353,0.269123,0.194769,0.189532,0.192459,0.195149
Al_Do,0.294560,0.307323,0.312139,0.301693,0.304858,0.307179,0.309091,0.312324,0.304005
Co_Do,0.214120,0.205282,0.187283,0.192261,0.120603,0.191430,0.191736,0.183455,0.196842
Al,0.589120,0.614046,0.624277,0.603386,0.610274,0.613801,0.618733,0.624086,0.608009
Co,0.410880,0.385954,0.375723,0.396614,0.389726,0.386199,0.381267,0.375914,0.391991
Al_Co_dist,0.318015,0.349974,0.361539,0.407332,0.295165,0.151701,0.226160,0.290595,0.175162
H,0.348723,0.314272,0.300926,0.328657,0.319305,0.314597,0.308103,0.301172,0.322356
Al_Up_η,0.317617,0.269882,0.287315,0.335510,0.324320,0.292515,0.364823,0.344725,0.303759
Co_Up_η,0.243811,0.330731,0.296949,0.226988,0.322371,0.291804,0.247003,0.408039,0.287108


In [56]:
start = timeit.default_timer()
res5_rtbl = [uz.multfit_dist(pxchg5.loc[(h-1)*3600:h*3600], s, α, hours, h)
             for h in hour_range]
stop = timeit.default_timer()
print('Time Spent: ', round(stop - start), ' seconds')
df5_rrest = pd.concat(res4_rtbl, axis=0)

Time Spent:  732  seconds


In [57]:
df5_rrest.T

,1,2,3,4,5,6,7,8,9
Al_Up,0.294560,0.306723,0.312139,0.301693,0.305416,0.306622,0.309642,0.311761,0.304005
Co_Up,0.196759,0.180672,0.188439,0.204353,0.269123,0.194769,0.189532,0.192459,0.195149
Al_Do,0.294560,0.307323,0.312139,0.301693,0.304858,0.307179,0.309091,0.312324,0.304005
Co_Do,0.214120,0.205282,0.187283,0.192261,0.120603,0.191430,0.191736,0.183455,0.196842
Al,0.589120,0.614046,0.624277,0.603386,0.610274,0.613801,0.618733,0.624086,0.608009
Co,0.410880,0.385954,0.375723,0.396614,0.389726,0.386199,0.381267,0.375914,0.391991
Al_Co_dist,0.318015,0.349974,0.361539,0.407332,0.295165,0.151701,0.226160,0.290595,0.175162
H,0.348723,0.314272,0.300926,0.328657,0.319305,0.314597,0.308103,0.301172,0.322356
Al_Up_η,0.317617,0.269882,0.287315,0.335510,0.324320,0.292515,0.364823,0.344725,0.303759
Co_Up_η,0.243811,0.330731,0.296949,0.226988,0.322371,0.291804,0.247003,0.408039,0.287108


In [ ]:
df0_rest['H'].plot(color='c');
df0_rrest['H'].plot(color='k');
df1_rrest['H'].plot(color='g');
df2_rrest['H'].plot(color='b');
df4_rrest['H'].plot(color='r');
df5_rrest['H'].plot(color='y');

In [ ]:
df0_rest['η'].plot(color='c');
df0_rrest['η'].plot(color='k');
df1_rrest['η'].plot(color='g');
df2_rrest['η'].plot(color='b');
df4_rrest['η'].plot(color='r');
df5_rrest['η'].plot(color='y');

In [ ]:
df_rest['σ'].plot(color='c');
df_rrest['σ'].plot(color='k');
df1_rrest['σ'].plot(color='g');
df2_rrest['σ'].plot(color='b');
df4_rrest['σ'].plot(color='r');
df5_rrest['σ'].plot(color='y');

In [ ]:
df0_rrest['μ'].plot(color='k');
df1_rrest['μ'].plot(color='g');
df2_rrest['μ'].plot(color='b');
df4_rrest['μ'].plot(color='r');
df5_rrest['μ'].plot(color='y');

In [ ]:
df4_rmrest['η'].plot(color='r');
df5_rmrest['η'].plot(color='y');

In [ ]:
df4_rmrest['σ'].plot(color='r');
df5_rmrest['σ'].plot(color='y');

In [ ]:
df4_rmrest['μ'].plot(color='r');
df5_rmrest['μ'].plot(color='y');

In [ ]:
uz.qq_plots(pxchg0, s, α, η, σ, μ, hours)

In [ ]:
uz.qq_plots(pxchg0, s, α, 0.327, 0.01, 0, hours)

In [ ]:
uz.cum_H(pxchg0).plot(figsize=(9, 6), color='k');
uz.cum_H(pxchg1).plot(figsize=(9, 6), color='g');
uz.cum_H(pxchg2).plot(figsize=(9, 6), color='b');
uz.cum_H(pxchg3).plot(figsize=(9, 6), color='m');
uz.cum_H(pxchg4).plot(figsize=(9, 6), color='r');

In [ ]:
uz.cum_H(pxchg0).loc[5000:].plot(figsize=(9, 6), color='k');
uz.cum_H(pxchg1).loc[5000:].plot(figsize=(9, 6), color='g');
uz.cum_H(pxchg2).loc[5000:].plot(figsize=(9, 6), color='b');
uz.cum_H(pxchg3).loc[5000:].plot(figsize=(9, 6), color='m');
uz.cum_H(pxchg4).loc[5000:].plot(figsize=(9, 6), color='r');

In [ ]:

def rolling_indic(df, window=100):
    dfc = df.copy()
    dfc['dPtj_r'] = dfc['dPtj'].rolling(window=window).sum()
    dfc['Co_r'] = dfc['Co'].rolling(window=window).sum()
    dfc['Al_r'] = dfc['Al'].rolling(window=window).sum()
    dfc['H_r'] = dfc['Co_r'] / (2 * dfc['Al_r'])
    dfc['dtj_Co'] = np.where(dfc['Co'], dfc['dtj'], 0)
    dfc['dtj_Al'] = np.where(dfc['Al'], dfc['dtj'], 0)
    dfc['dtj_Co_r'] = dfc['dtj_Co'].rolling(window=window).sum() / dfc['Co_r']
    dfc['dtj_Al_r'] = dfc['dtj_Al'].rolling(window=window).sum() / dfc['Al_r']
    dfc['dtj_Co/Al_r'] = dfc['dtj_Co_r'] / dfc['dtj_Al_r']
    dfc['eta_r'] = (np.sqrt(dfc['dtj_Co/Al_r']**2 - dfc['dtj_Co/Al_r'] + 1) +
                    (1 - dfc['dtj_Co/Al_r'])) / (2 * dfc['dtj_Co/Al_r'])
    return dfc

In [ ]:

window_roll = 500

pxchg0_r = rolling_indic(pxchg0, window_roll)
pxchg1_r = rolling_indic(pxchg1, window_roll)
pxchg2_r = rolling_indic(pxchg2, window_roll)
pxchg3_r = rolling_indic(pxchg3, window_roll)
pxchg4_r = rolling_indic(pxchg4, window_roll)

In [ ]:

pxchg0_r['dPtj_r'].plot(figsize=(9, 6), color='k');
pxchg1_r['dPtj_r'].plot(figsize=(9, 6), color='g');
pxchg2_r['dPtj_r'].plot(figsize=(9, 6), color='b');
pxchg3_r['dPtj_r'].plot(figsize=(9, 6), color='m');
pxchg4_r['dPtj_r'].plot(figsize=(9, 6), color='r');

In [ ]:

pxchg0_r['H_r'].plot(figsize=(9, 6), color='k');
pxchg1_r['H_r'].plot(figsize=(9, 6), color='g');
pxchg2_r['H_r'].plot(figsize=(9, 6), color='b');
pxchg3_r['H_r'].plot(figsize=(9, 6), color='m');
pxchg4_r['H_r'].plot(figsize=(9, 6), color='r');

In [ ]:

pxchg0_r['dtj_Al_r'].plot(figsize=(9, 6), color='k');
pxchg1_r['dtj_Al_r'].plot(figsize=(9, 6), color='g');
pxchg2_r['dtj_Al_r'].plot(figsize=(9, 6), color='b');
pxchg3_r['dtj_Al_r'].plot(figsize=(9, 6), color='m');
pxchg4_r['dtj_Al_r'].plot(figsize=(9, 6), color='r');

In [ ]:

pxchg0_r['dtj_Co_r'].plot(figsize=(9, 6), color='k');
pxchg1_r['dtj_Co_r'].plot(figsize=(9, 6), color='g');
pxchg2_r['dtj_Co_r'].plot(figsize=(9, 6), color='b');
pxchg3_r['dtj_Co_r'].plot(figsize=(9, 6), color='m');
pxchg4_r['dtj_Co_r'].plot(figsize=(9, 6), color='r');

In [ ]:

(pxchg0_r['dtj_Co_r'] / pxchg0_r['dtj_Al_r']).plot(figsize=(9, 6), color='k');
(pxchg1_r['dtj_Co_r'] / pxchg1_r['dtj_Al_r']).plot(figsize=(9, 6), color='g');
(pxchg2_r['dtj_Co_r'] / pxchg2_r['dtj_Al_r']).plot(figsize=(9, 6), color='b');
(pxchg3_r['dtj_Co_r'] / pxchg3_r['dtj_Al_r']).plot(figsize=(9, 6), color='m');
(pxchg4_r['dtj_Co_r'] / pxchg4_r['dtj_Al_r']).plot(figsize=(9, 6), color='r');

In [ ]:

pxchg0_r['eta_r'].plot(figsize=(9, 6), color='k', linestyle=':');
pxchg1_r['eta_r'].plot(figsize=(9, 6), color='g');
pxchg2_r['eta_r'].plot(figsize=(9, 6), color='b');
pxchg3_r['eta_r'].plot(figsize=(9, 6), color='m');
pxchg4_r['eta_r'].plot(figsize=(9, 6), color='r', linestyle=':');

In [ ]:

algo_tr = rpt.Pelt(model='l2', min_size=100).fit(pxchg2['dtj'].dropna().values)
my_bkps_tr = algo_tr.predict(pen=3)

In [ ]:

rpt.display(pxchg2['dtj'].dropna().values, my_bkps_tr)


